# Construction et audit de la couche Bronze

## Objectif

Ce notebook construit la couche Bronze à partir des exports CSV legacy, compare plusieurs organisations physiques en Parquet, mesure leurs coûts et leurs performances, puis déploie dans le bucket du groupe uniquement les layouts retenus.

Le Bronze doit conserver les données reçues :

- aucune suppression de colonne ;
- aucune déduplication ;
- aucune correction de valeur ;
- aucune suppression de ligne ;
- compression sans perte ;
- conservation des valeurs sources.

Les éventuelles colonnes techniques dérivées ou jointures entre tables ne seront utilisées dans le Bronze que si cette pratique est validée par l’enseignant. Dans le cas contraire, elles seront reportées à la couche Silver.

---

## Workflow

Le notebook suit les étapes suivantes :

1. chargement des CSV bruts dans `raw_tables` ;
2. copie des données dans `prepared_tables` ;
3. création éventuelle de colonnes techniques de partitionnement ;
4. exploration des colonnes candidates ;
5. lecture des layouts à tester depuis `explore_data_mapping.json` ;
6. génération locale des différents layouts Parquet ;
7. mesure du nombre d’objets, de leur taille et du temps d’écriture ;
8. upload des layouts d’audit dans le bucket personnel ;
9. création de tables externes BigQuery ;
10. exécution de requêtes `SELECT` représentatives ;
11. comparaison des coûts et performances ;
12. choix d’un seul layout final par table ;
13. déploiement des fichiers retenus dans le bucket Bronze du groupe ;
14. vérification du contenu local par rapport au contenu GCS.

Les fichiers placés sous `audit/partition_tests/` sont temporaires et ne constituent pas le Bronze final.

---

# Préparation des données

## Données brutes

Les CSV sont chargés une seule fois :

```text
CSV sources → raw_tables
```

`raw_tables` représente les données reçues et ne doit jamais être modifié.

Une copie de travail est ensuite créée :

```text
raw_tables → prepared_tables
```

Les layouts Parquet sont générés à partir de `prepared_tables`.

## Transformations techniques éventuelles

Certaines partitions temporelles nécessitent une colonne moins granulaire :

```text
order_date → order_month
created_at → payment_month
deal_date → deal_month
est_delivery_date → delivery_month
start_date → start_month
```

Certaines tables peuvent également être enrichies par une jointure contrôlée :

```text
order_items + orders → order_month
order_items + product → category_id
shipment + product → category_id
```

Toute jointure utilisée dans cette phase doit :

- être une jointure gauche ;
- utiliser une relation `many-to-one` validée ;
- conserver exactement le même nombre de lignes ;
- ne supprimer aucune colonne source ;
- ne créer aucun doublon ;
- échouer explicitement si une clé ne correspond pas.

Ces transformations sont conditionnelles à la validation de leur compatibilité avec la définition du Bronze.

---

# Exploration des colonnes candidates

L’exploration permet d’identifier les colonnes pouvant servir de clés de partitionnement.

Les couleurs constituent seulement une première indication. La décision finale dépend également :

- des filtres réellement utilisés ;
- du volume attendu en production ;
- du nombre de fichiers produits ;
- de leur taille ;
- du coût de stockage et d’écriture ;
- des performances BigQuery.

## Métriques d’exploration

Pour chaque table :

$$
\text{rows} = \text{nombre total de lignes}
$$

Pour chaque colonne :

$$
\text{non-null values}
=
\text{rows} - \text{null count}
$$

$$
\text{unique count}
=
\text{nombre de valeurs distinctes non nulles}
$$

$$
\text{unique ratio}
=
\frac{\text{unique count}}
{\text{non-null values}}
$$

Le ratio d’unicité estime la proportion de valeurs distinctes dans la colonne. Il donne une première indication du nombre potentiel de partitions, mais il ne suffit pas à prendre une décision.

---

## Règles visuelles

### Taille de la table

| Condition | Couleur | Interprétation |
|---|---|---|
| `rows < 20` | Jaune | Table trop petite pour fournir une comparaison quantitative fiable |
| `rows >= 20` | Aucune | Table pouvant être étudiée normalement |

Une petite table reste généralement non partitionnée.

### Valeurs manquantes

| Condition | Couleur | Interprétation |
|---|---|---|
| `null_count = 0` | Aucune | Colonne exploitable dans l’audit |
| `null_count > 0` | Rouge | Risque de partitions nulles ou ambiguës |

Dans cet audit, une colonne contenant des valeurs manquantes est écartée de la présélection automatique.

### Nombre absolu de valeurs distinctes

| Condition | Couleur | Interprétation |
|---|---|---|
| `unique_count <= 1 000` | Aucune | Nombre de groupes encore contrôlable |
| `unique_count > 1 000` | Rouge | Risque important de surpartitionnement |

Le nombre absolu de valeurs distinctes est important : un ratio faible peut malgré tout produire plusieurs milliers de partitions sur une grande table.

### Ratio de valeurs distinctes

| Ratio | Couleur | Interprétation |
|---|---|---|
| `< 5 %` | Vert foncé | Faible cardinalité et nombre limité de groupes |
| `5 % – 40 %` | Vert clair | Cardinalité potentiellement exploitable |
| `> 40 % – 75 %` | Jaune | À étudier selon le volume et les requêtes |
| `> 75 %` | Rouge | Risque élevé de nombreux petits fichiers |

Une très faible cardinalité n’implique pas automatiquement un bon partitionnement. Une colonne binaire peu utilisée dans les filtres reste une mauvaise clé.

---

## Critères métier

Une clé de partitionnement pertinente doit idéalement :

1. apparaître fréquemment dans les clauses `WHERE` ;
2. être stable dans le temps ;
3. avoir une cardinalité contrôlée ;
4. produire des partitions suffisamment volumineuses et équilibrées ;
5. réduire les données lues par BigQuery ;
6. éviter la création d’un grand nombre de petits fichiers.

Les identifiants uniques, les mesures numériques continues et les champs rarement filtrés sont généralement de mauvais candidats.

---

# Configuration des layouts d’audit

Le fichier `explore_data_mapping.json` contient les layouts à comparer.

Exemple :

```json
{
  "orders": [
    null,
    ["order_month"]
  ],
  "shipment": [
    null,
    ["delivery_month"],
    ["delivery_month", "status"]
  ]
}
```

Signification :

- `null` : Parquet non partitionné ;
- `["order_month"]` : partition simple ;
- `["delivery_month", "status"]` : partition hiérarchique, d’abord par mois puis par statut.

Chaque table doit posséder une version non partitionnée servant de référence.

---

# Génération locale des layouts

Chaque layout est écrit en Parquet avec compression Snappy.

Pour chaque layout, les métriques locales suivantes sont calculées :

$$
\text{partition count}
=
\text{nombre de combinaisons distinctes des clés}
$$

$$
\text{object count}
=
\text{nombre de fichiers Parquet produits}
$$

$$
\text{total bytes}
=
\sum \text{taille des fichiers Parquet}
$$

$$
\text{average file bytes}
=
\frac{\text{total bytes}}
{\text{object count}}
$$

Le temps d’écriture local est également mesuré :

$$
\text{local write seconds}
=
t_{\text{fin écriture}} - t_{\text{début écriture}}
$$

Cette durée dépend de la machine locale. Elle sert principalement à détecter les layouts produisant un nombre excessif de fichiers.

---

# Upload des layouts d’audit dans GCS

Les layouts sont envoyés dans le bucket personnel sous un préfixe propre à chaque exécution :

```text
audit/partition_tests/<run_id>/<table>/<layout>/
```

Pour chaque layout, l’audit mesure :

- le nombre d’objets réellement présents dans GCS ;
- le nombre total d’octets ;
- le temps d’upload ;
- la correspondance entre les fichiers locaux et les objets GCS.

Le temps d’upload est une métrique diagnostique. Il dépend du réseau et ne constitue pas directement une mesure du temps d’exécution BigQuery.

---

# Coûts GCS

## Coût mensuel de stockage

Avec un tarif Standard de `0.020 USD/GiB-mois` :

$$
\text{storage cost}
=
\frac{\text{total bytes}}
{1024^3}
\times 0.020
$$

## Coût de création des objets

Avec un tarif de classe A de `0.005 USD pour 1 000 opérations` :

$$
\text{Class A write cost}
=
\frac{\text{object count}}
{1000}
\times 0.005
$$

Ce coût représente la création des objets Parquet dans GCS. Il ne représente pas le coût d’un `SELECT` BigQuery.

Si l’ingestion complète est répétée chaque mois, ce coût d’écriture peut être considéré comme un coût mensuel d’ingestion. Sinon, il s’agit d’un coût ponctuel.

---

# Tests BigQuery

Une table externe BigQuery est créée pour chaque layout présent dans GCS.

Les fichiers restent dans Cloud Storage : ils ne sont pas chargés dans une table BigQuery native.

## Requêtes testées

L’audit exécute des requêtes `SELECT` représentatives, généralement sous la forme :

```sql
SELECT COUNT(*)
FROM `project.dataset.external_table`
WHERE partition_column = value;
```

Pour assurer une comparaison correcte :

- tous les layouts d’une même table utilisent le même DataFrame ;
- la même logique de requête est appliquée à tous les layouts ;
- le cache BigQuery est désactivé ;
- la requête contient un filtre correspondant à l’usage métier étudié ;
- aucun `INSERT`, `UPDATE` ou `DELETE` n’est testé.

## Temps d’exécution

$$
\text{query elapsed ms}
=
t_{\text{fin du job}}
-
t_{\text{début du job}}
$$

`query_elapsed_ms` mesure la durée réelle du job BigQuery.

Sur les petits échantillons, cette mesure peut être dominée par le temps de démarrage du moteur. Elle doit donc être interprétée avec prudence.

## Données traitées

$$
\text{query bytes processed}
=
\text{nombre d’octets parcourus par BigQuery}
$$

`query_bytes_processed` permet d’évaluer si le partitionnement réduit réellement le volume lu.

Cette métrique constitue un indicateur du coût des requêtes BigQuery. Sa conversion en dollars nécessite le tarif BigQuery applicable et le nombre mensuel de requêtes, qui ne sont pas inclus dans les deux constantes de coût utilisées dans cet exercice.

---

# Métriques finales de comparaison

Les décisions sont principalement fondées sur :

| Métrique | Service | Interprétation |
|---|---|---|
| `gcs_object_count` | GCS | Nombre de fichiers Parquet créés |
| `gcs_total_bytes` | GCS | Taille totale du layout |
| `average_file_bytes` | Local/GCS | Taille moyenne des objets |
| `class_a_write_cost_usd` | GCS | Coût de création des objets |
| `storage_cost_usd_month` | GCS | Coût mensuel du stockage |
| `query_elapsed_ms` | BigQuery | Temps d’exécution du `SELECT` |
| `query_bytes_processed` | BigQuery | Volume traité par la requête |

Les métriques suivantes sont conservées comme informations diagnostiques :

- `partition_count` ;
- `local_write_seconds` ;
- `upload_seconds`.

---

# Limites de l’audit

Certaines tables sont vides ou ne contiennent qu’un très petit nombre de lignes.

Pour ces tables :

- aucun gain de performance ne peut être démontré ;
- un Parquet vide non partitionné peut conserver le schéma ;
- un partitionnement physique ne peut pas être matérialisé sans valeur de partition ;
- une future clé de partition peut être documentée comme décision théorique ;
- la décision doit être fondée sur la volumétrie attendue et les usages métier.

Les tables de quelques dizaines ou centaines de milliers de lignes peuvent également être trop petites pour faire apparaître un gain de temps stable dans BigQuery.

L’audit permet néanmoins de détecter :

- la surpartition ;
- la multiplication des objets ;
- l’augmentation du stockage ;
- les partitions trop petites ;
- les layouts ne réduisant pas les octets traités.

---

# Protection des données sensibles

L’exploration ne doit jamais afficher d’exemples réels pour les colonnes sensibles telles que :

- mots de passe ;
- numéros de carte ;
- CVV ;
- adresses ;
- numéros de téléphone ;
- adresses électroniques.

Ces colonnes doivent être identifiées dans le dictionnaire de données et leurs exemples doivent être masqués dans les sorties du notebook.

Le bucket Bronze ne doit pas être public et son accès doit suivre le principe du moindre privilège.

---

# Choix du layout final

Le fichier `final_bronze_layout.json` contient exactement une décision par table :

```json
{
  "orders": {
    "layout": "by_order_month",
    "reason": "Queries frequently filter orders by accounting period"
  }
}
```

La décision finale tient compte :

- des mesures de l’audit ;
- des usages métier ;
- de la volumétrie actuelle ;
- de la volumétrie attendue ;
- de la stabilité de la clé ;
- du risque de petits fichiers ;
- des limites des échantillons disponibles.

Il est normal que certaines tables restent non partitionnées.

---

# Déploiement dans le Bronze du groupe

Seuls les layouts sélectionnés sont transférés dans le bucket du groupe :

```text
bronze/<table>/<partition folders>/<parquet files>
```

Le déploiement reprend les fichiers exacts qui ont été testés pendant l’audit.

Pour chaque table, la validation compare :

- la liste des fichiers locaux ;
- la liste des objets GCS ;
- le nombre d’objets ;
- la taille de chaque objet ;
- le volume total.

Le déploiement est validé uniquement si aucun objet ne manque, si aucun objet inattendu n’est présent et si toutes les tailles correspondent.

Le coût présenté dans `docs/costs.md` doit être calculé à partir du Bronze final uniquement, et non à partir de l’ensemble des layouts temporaires de l’audit.

Les mesures obtenues alimenteront également :

- `docs/adr/0002-bronze-layout.md` ;
- `docs/data-dictionary.md` ;
- `infra/lifecycle.json` ;
- `infra/iam.md` ;
- `docs/architecture.md`.

In [24]:
from pathlib import Path
from datetime import datetime
from time import perf_counter
import hashlib
import json
import re
import pyarrow.parquet as pq
import base64
import pandas as pd
from urllib.parse import quote
from IPython.display import display

import google.auth
from google.cloud import storage, bigquery
from google.api_core.exceptions import NotFound


# ============================================================
# PROJET
# ============================================================

PROJECT_ID = "data-engineering-a"
GCP_REGION = "europe-west1"

# Bucket personnel utilisé uniquement pour l'audit
AUDIT_BUCKET_NAME = "data-engineering-a-alexandro"
AUDIT_PREFIX = "audit/partition_tests"

# Dataset contenant les tables externes d'audit
BIGQUERY_DATASET = "alexandro_dataset_bronze"

# À remplacer par le vrai nom du bucket du groupe
GROUP_BUCKET_NAME = "data-engineering-a-groupe-1"
FINAL_BRONZE_PREFIX = "bronze"


# ============================================================
# CHEMINS LOCAUX
# ============================================================

INGESTION_DIR = Path.cwd()

if INGESTION_DIR.name != "ingestion":
    raise RuntimeError(
        "Le notebook doit être exécuté depuis le dossier ingestion."
    )

SOURCE_DIR = Path("../../legacy_csv").resolve()

AUDIT_LOCAL_DIR = (
    INGESTION_DIR / "audit_layout"
).resolve()

AUDIT_CONFIG_PATH = (
    INGESTION_DIR / "explore_data_mapping.json"
).resolve()

FINAL_LAYOUT_CONFIG_PATH = (
    INGESTION_DIR / "final_bronze_layout.json"
).resolve()

RESULTS_DIR = (
    INGESTION_DIR / "audit_results"
).resolve()

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = AUDIT_LOCAL_DIR / RUN_ID


# ============================================================
# TARIFS DU COURS
# ============================================================

STANDARD_STORAGE_USD_PER_GIB_MONTH = 0.020
CLASS_A_USD_PER_1000_OPERATIONS = 0.005
BYTES_PER_GIB = 1024 ** 3


# ============================================================
# AUTHENTIFICATION
# ============================================================

credentials, detected_project = google.auth.default()

storage_client = storage.Client(
    project=PROJECT_ID,
    credentials=credentials,
)

bigquery_client = bigquery.Client(
    project=PROJECT_ID,
    credentials=credentials,
)


# ============================================================
# VÉRIFICATIONS
# ============================================================

print("Projet configuré :", PROJECT_ID)
print("Projet ADC détecté :", detected_project)
print("Source CSV :", SOURCE_DIR)
print("Sortie locale audit :", RUN_DIR)
print("Configuration audit :", AUDIT_CONFIG_PATH)
print("Configuration finale :", FINAL_LAYOUT_CONFIG_PATH)


if not SOURCE_DIR.exists():
    raise RuntimeError(
        f"Dossier CSV introuvable : {SOURCE_DIR}"
    )


# Bucket personnel
audit_bucket = storage_client.get_bucket(
    AUDIT_BUCKET_NAME
)

print(
    "Bucket d'audit :",
    audit_bucket.name,
    "| région :",
    audit_bucket.location,
)


# Dataset BigQuery
dataset_id = f"{PROJECT_ID}.{BIGQUERY_DATASET}"
dataset = bigquery_client.get_dataset(dataset_id)

print(
    "Dataset BigQuery :",
    dataset_id,
    "| région :",
    dataset.location,
)


if dataset.location.lower() != GCP_REGION.lower():
    raise RuntimeError(
        f"Le dataset est dans {dataset.location}, "
        f"pas dans {GCP_REGION}."
    )


# Bucket final
if GROUP_BUCKET_NAME == "REPLACE_WITH_GROUP_BUCKET_NAME":
    print(
        "Bucket du groupe non configuré : "
        "remplace GROUP_BUCKET_NAME avant l'upload final."
    )
else:
    group_bucket = storage_client.get_bucket(
        GROUP_BUCKET_NAME
    )

    print(
        "Bucket du groupe :",
        group_bucket.name,
        "| région :",
        group_bucket.location,
    )

    if group_bucket.location.lower() != GCP_REGION.lower():
        raise RuntimeError(
            f"Le bucket du groupe est dans "
            f"{group_bucket.location}, pas dans {GCP_REGION}."
        )

Projet configuré : data-engineering-a
Projet ADC détecté : data-engineering-a
Source CSV : /Users/alexandro/Desktop/Master/Mines Paris/M2/Data Management & Cloud infrastructure for AI /legacy_csv
Sortie locale audit : /Users/alexandro/Desktop/Master/Mines Paris/M2/Data Management & Cloud infrastructure for AI /de-platform-template/ingestion/audit_layout/20261012_121604
Configuration audit : /Users/alexandro/Desktop/Master/Mines Paris/M2/Data Management & Cloud infrastructure for AI /de-platform-template/ingestion/explore_data_mapping.json
Configuration finale : /Users/alexandro/Desktop/Master/Mines Paris/M2/Data Management & Cloud infrastructure for AI /de-platform-template/ingestion/final_bronze_layout.json
Bucket d'audit : data-engineering-a-alexandro | région : EUROPE-WEST1
Dataset BigQuery : data-engineering-a.alexandro_dataset_bronze | région : europe-west1
Bucket du groupe : data-engineering-a-groupe-1 | région : EUROPE-WEST1


### 1. chargement des CSV bruts dans `raw_tables` 

In [3]:
csv_files = sorted(SOURCE_DIR.glob("*.csv"))

if not csv_files:
    raise RuntimeError(
        f"Aucun fichier CSV trouvé dans : {SOURCE_DIR}"
    )

raw_tables = {}

for csv_path in csv_files:
    table_name = csv_path.stem

    

    try:
        df = pd.read_csv(csv_path)
    except Exception as error:
        raise RuntimeError(
            f"Impossible de lire {csv_path.name}"
        ) from error

    raw_tables[table_name] = df



# Vérification générale
if len(raw_tables) != len(csv_files):
    raise RuntimeError(
        "Tous les CSV n'ont pas été chargés."
    )


raw_inventory_df = pd.DataFrame([
    {
        "table": table_name,
        "rows": len(df),
        "columns": len(df.columns),
        "memory_bytes": int(
            df.memory_usage(
                index=True,
                deep=True,
            ).sum()
        ),
        "is_empty": df.empty,
    }
    for table_name, df in raw_tables.items()
]).sort_values("table").reset_index(drop=True)


display(
    raw_inventory_df.style.format({
        "rows": "{:,.0f}",
        "columns": "{:,.0f}",
        "memory_bytes": "{:,.0f}",
    })
)


print(
    "\nChargement terminé :",
    len(raw_tables),
    "table(s)"
)

print(
    "Nombre total de lignes :",
    f"{raw_inventory_df['rows'].sum():,}"
)

print(
    "Tables vides :",
    raw_inventory_df.loc[
        raw_inventory_df["is_empty"],
        "table",
    ].tolist()
)

,table,rows,columns,memory_bytes,is_empty
0,buyer,"100,000",2,"9,300,156",False
1,carrier,4,3,495,False
2,cart,0,5,132,True
3,cart_items,0,4,132,True
4,category,2,4,444,False
5,customer,"130,766",7,"49,691,542",False
6,customer_payment,"100,000",4,"10,900,156",False
7,customer_shipping,"200,001",4,"21,800,275",False
8,daily_deals,0,4,132,True
9,discount,0,5,132,True



Chargement terminé : 27 table(s)
Nombre total de lignes : 2,928,485
Tables vides : ['cart', 'cart_items', 'daily_deals', 'discount', 'returns']


### 2. copie des données dans prepared_tables 

In [10]:
prepared_tables = {
    table_name: df.copy(deep=True)
    for table_name, df in raw_tables.items()
}


# Vérification que toutes les tables ont été copiées
if set(prepared_tables) != set(raw_tables):
    raise RuntimeError(
        "Les tables de prepared_tables ne correspondent pas "
        "aux tables de raw_tables."
    )


copy_validation = []

for table_name in sorted(raw_tables):
    raw_df = raw_tables[table_name]
    prepared_df = prepared_tables[table_name]

    same_rows = len(raw_df) == len(prepared_df)

    same_columns = (
        raw_df.columns.tolist()
        == prepared_df.columns.tolist()
    )

    same_values = raw_df.equals(prepared_df)

    independent_object = (
        id(raw_df) != id(prepared_df)
    )

    copy_validation.append({
        "table": table_name,
        "rows": len(prepared_df),
        "same_rows": same_rows,
        "same_columns": same_columns,
        "same_values": same_values,
        "independent_object": independent_object,
        "copy_valid": (
            same_rows
            and same_columns
            and same_values
            and independent_object
        ),
    })


copy_validation_df = pd.DataFrame(
    copy_validation
)


# display(
#     copy_validation_df.style.applymap(
#         lambda value: (
#             "background-color: #b7e4c7; color: #081c15"
#             if value is True
#             else (
#                 "background-color: #ffadad; color: #450920"
#                 if value is False
#                 else ""
#             )
#         ),
#         subset=[
#             "same_rows",
#             "same_columns",
#             "same_values",
#             "independent_object",
#             "copy_valid",
#         ],
#     )
# )


if not copy_validation_df["copy_valid"].all():
    invalid_tables = copy_validation_df.loc[
        ~copy_validation_df["copy_valid"],
        "table",
    ].tolist()

    raise RuntimeError(
        f"Copies invalides : {invalid_tables}"
    )


print(
    "Copie terminée :",
    len(prepared_tables),
    "table(s) disponibles dans prepared_tables."
)

Copie terminée : 27 table(s) disponibles dans prepared_tables.


### 3. création éventuelle de colonnes techniques de partitionnement 

| Priorité | Type | Table produite | Transformation | Partition proposée |
|---|---|---|---|---|
| Forte | Intra-table | `orders` | `order_date` → mois | `order_month` |
| Forte | Intra-table | `daily_deals` | `deal_date` → mois | `deal_month` |
| Forte | Intra-table | `subscription` | `start_date` → mois | `start_month` |
| Forte | Inter-table | `order_items` + `orders` | Récupération de `order_date`, puis conversion mensuelle | `order_month` |
| Forte | Inter-table | `order_items` + `orders` + `product` | Ajout du mois de commande et de la catégorie | `order_month`, `category_id` |
| Moyenne | Inter-table | `daily_deals` + `product` | Ajout de la catégorie du produit | `deal_month`, `category_id` |
| Moyenne | Inter-table | `shipment` + `product` | Ajout de la catégorie du produit | `delivery_month`, `category_id` |
| Faible | Inter-table | `payment` + `orders` | Ajout du mois de commande | `order_month`, `method` |
| Faible | Inter-table | `seller_products` + `seller` + `product` | Ajout du type de vendeur et de la catégorie | `seller_type`, `category_id` |

### 4. exploration des colonnes candidates 

In [11]:
SENSITIVE_COLUMNS = {
    "fname", "lname", "phone", "email", "pwd", "password",
    "card_no", "cvv", "expiry_date", "billing_address", "address"
}


def is_sensitive_column(column):
    name = column.lower()
    return name in SENSITIVE_COLUMNS or any(word in name for word in ["password", "card", "cvv", "phone", "email", "address"])


def explore_dataframe(table_name, df):
    total_rows = len(df)
    source_columns = set(raw_tables[table_name].columns)
    results = []

    for column in df.columns:
        non_null = int(df[column].notna().sum())
        null_count = int(df[column].isna().sum())
        unique_count = int(df[column].nunique(dropna=True))
        unique_ratio = unique_count / non_null if non_null else pd.NA
        sample_values = "[REDACTED]" if is_sensitive_column(column) else df[column].dropna().unique()[:2].tolist()

        results.append({
            "table": table_name,
            "column": column,
            "origin": "source" if column in source_columns else "derived",
            "dtype": str(df[column].dtype),
            "rows": total_rows,
            "non_null": non_null,
            "null_count": null_count,
            "unique_count": unique_count,
            "unique_ratio": unique_ratio,
            "sample_values": sample_values,
        })

    return pd.DataFrame(results)


def color_unique_ratio(value):
    if pd.isna(value):
        return ""
    if value < 0.05:
        return "background-color: #2d6a4f; color: #ffffff"
    if value <= 0.40:
        return "background-color: #b7e4c7; color: #081c15"
    if value <= 0.75:
        return "background-color: #ffe699; color: #332701"
    return "background-color: #ffadad; color: #450920"


def color_rows(value):
    return "background-color: #ffe699; color: #332701" if value < 20 else ""


def color_null_count(value):
    return "background-color: #ffadad; color: #450920" if value > 0 else ""


def color_unique_count(value):
    return "background-color: #ff4d4d; color: #ffffff" if value > 1000 else ""


def color_origin(value):
    return "background-color: #e1bee7; color: #4a148c; font-weight: bold" if value == "derived" else ""


highlight_columns = {
    "background-color": "#e3f2fd",
    "border-left": "2px solid #1976d2",
    "border-right": "2px solid #1976d2",
    "font-weight": "bold",
}

all_results = []

for table_name, df in sorted(prepared_tables.items()):
    result = explore_dataframe(table_name, df)
    all_results.append(result)

    print(f"\n{table_name}")

    styled_result = (
        result.style
        .set_properties(subset=["rows", "null_count", "unique_count"], **highlight_columns)
        .set_table_styles([
            {"selector": "th.col_heading.level0.col4", "props": list(highlight_columns.items())},
            {"selector": "th.col_heading.level0.col6", "props": list(highlight_columns.items())},
            {"selector": "th.col_heading.level0.col7", "props": list(highlight_columns.items())},
        ])
        .applymap(color_origin, subset=["origin"])
        .applymap(color_unique_ratio, subset=["unique_ratio"])
        .applymap(color_null_count, subset=["null_count"])
        .applymap(color_rows, subset=["rows"])
        .applymap(color_unique_count, subset=["unique_count"])
        .format({
            "rows": "{:,.0f}",
            "non_null": "{:,.0f}",
            "null_count": "{:,.0f}",
            "unique_count": "{:,.0f}",
            "unique_ratio": lambda value: "" if pd.isna(value) else f"{value:.1%}",
        })
    )

    display(styled_result)

audit_df = pd.concat(all_results, ignore_index=True)

print("\nTables explorées :", audit_df["table"].nunique())
print("Colonnes explorées :", len(audit_df))
print("Colonnes dérivées :", int((audit_df["origin"] == "derived").sum()))


buyer


,table,column,origin,dtype,rows,non_null,null_count,unique_count,unique_ratio,sample_values
0,buyer,Unnamed: 0,source,int64,"100,000","100,000",0,"100,000",100.0%,"[0, 1]"
1,buyer,buyer_id,source,object,"100,000","100,000",0,"100,000",100.0%,"['AE222BBOVZIF42YOOPNBXL4UUMYA', 'AE222FP7YRNFCEQ2W3ZDIGMSYTLQ']"



carrier


,table,column,origin,dtype,rows,non_null,null_count,unique_count,unique_ratio,sample_values
0,carrier,Unnamed: 0,source,int64,4,4,0,4,100.0%,"[0, 1]"
1,carrier,carrier_id,source,int64,4,4,0,4,100.0%,"[1, 2]"
2,carrier,carrier_name,source,object,4,4,0,4,100.0%,"['Albatros Parcels', 'Blue Heron Logistics']"



cart


...


subscription


,table,column,origin,dtype,rows,non_null,null_count,unique_count,unique_ratio,sample_values
0,subscription,Unnamed: 0,source,int64,"50,091","50,091",0,"50,091",100.0%,"[0, 1]"
1,subscription,subscription_id,source,int64,"50,091","50,091",0,"50,091",100.0%,"[1, 2]"
2,subscription,c_id,source,object,"50,091","50,091",0,"50,091",100.0%,"['AE222Y4WTST6BUZ4J5Y2H6QMBITQ', 'AE22355IAQGYY4EPWWVYX25J7AHA']"
3,subscription,start_date,source,object,"50,091","50,091",0,366,0.7%,"['2025-03-02', '2025-02-25']"
4,subscription,end_date,source,object,"50,091","50,091",0,366,0.7%,"['2026-03-02', '2026-02-25']"



wishlist_item


,table,column,origin,dtype,rows,non_null,null_count,unique_count,unique_ratio,sample_values
0,wishlist_item,Unnamed: 0,source,int64,"24,810","24,810",0,"24,810",100.0%,"[0, 1]"
1,wishlist_item,product_id,source,object,"24,810","24,810",0,"18,842",75.9%,"['0816091846', '0985393300']"
2,wishlist_item,buyer_id,source,object,"24,810","24,810",0,"19,823",79.9%,"['AEIY4E2ZRDXCETPZJCOIP6QUS2GQ', 'AEKXN4RB2KFNZHKC3K6ETHDA3MCA']"



Tables explorées : 27
Colonnes explorées : 128
Colonnes dérivées : 0


### 5. lecture des layouts à tester depuis explore_data_mapping.json 

In [13]:
MAX_AUDIT_PARTITIONS = 3000


def make_layout_name(partition_columns):
    return "unpartitioned" if partition_columns is None else "by_" + "_then_".join(partition_columns)


if not AUDIT_CONFIG_PATH.exists():
    raise FileNotFoundError(f"Configuration introuvable : {AUDIT_CONFIG_PATH}")

with open(AUDIT_CONFIG_PATH, "r", encoding="utf-8") as file:
    requested_partition_config = json.load(file)

if not isinstance(requested_partition_config, dict):
    raise TypeError("Le fichier JSON doit contenir un objet associant chaque table à une liste de layouts.")


configured_tables = set(requested_partition_config)
prepared_table_names = set(prepared_tables)

missing_tables = prepared_table_names - configured_tables
unknown_tables = configured_tables - prepared_table_names

if missing_tables:
    raise ValueError(f"Tables absentes du JSON : {sorted(missing_tables)}")

if unknown_tables:
    raise ValueError(f"Tables inconnues dans le JSON : {sorted(unknown_tables)}")


partition_config = {}
layout_validation = []

for table_name, candidates in requested_partition_config.items():
    if not isinstance(candidates, list):
        raise TypeError(f"{table_name} doit être associé à une liste.")

    if None not in candidates:
        raise ValueError(f"{table_name} ne contient pas le layout null de référence.")

    df = prepared_tables[table_name]
    accepted_candidates = []
    seen_candidates = set()

    for partition_columns in candidates:
        if partition_columns is not None:
            if not isinstance(partition_columns, list) or not partition_columns:
                raise TypeError(f"Layout invalide pour {table_name} : {partition_columns}")

            if not all(isinstance(column, str) for column in partition_columns):
                raise TypeError(f"Les colonnes de {table_name} doivent être des chaînes de caractères.")

            if len(partition_columns) != len(set(partition_columns)):
                raise ValueError(f"Colonne répétée dans {table_name} : {partition_columns}")

        signature = None if partition_columns is None else tuple(partition_columns)

        if signature in seen_candidates:
            raise ValueError(f"Layout dupliqué pour {table_name} : {partition_columns}")

        seen_candidates.add(signature)
        layout_name = make_layout_name(partition_columns)
        status = "ACCEPTED"
        reason = ""
        estimated_partitions = 1

        if partition_columns is not None:
            missing_columns = [column for column in partition_columns if column not in df.columns]

            if missing_columns:
                raise ValueError(f"Colonnes absentes de {table_name} : {missing_columns}")

            if df.empty:
                status = "SKIPPED"
                reason = "empty table"
                estimated_partitions = 0
            else:
                null_counts = df[partition_columns].isna().sum()
                columns_with_nulls = null_counts[null_counts > 0].to_dict()

                if columns_with_nulls:
                    status = "SKIPPED"
                    reason = f"null values: {columns_with_nulls}"
                else:
                    estimated_partitions = len(df[partition_columns].drop_duplicates())

                    if estimated_partitions > MAX_AUDIT_PARTITIONS:
                        status = "SKIPPED"
                        reason = f"{estimated_partitions:,} estimated partitions exceeds limit of {MAX_AUDIT_PARTITIONS:,}"

        if status == "ACCEPTED":
            accepted_candidates.append(partition_columns)

        layout_validation.append({
            "table": table_name,
            "layout": layout_name,
            "partition_columns": partition_columns,
            "rows": len(df),
            "estimated_partitions": estimated_partitions,
            "status": status,
            "reason": reason,
        })

    partition_config[table_name] = accepted_candidates


layout_validation_df = pd.DataFrame(layout_validation)

display(
    layout_validation_df.style
    .applymap(
        lambda value: (
            "background-color: #b7e4c7; color: #081c15"
            if value == "ACCEPTED"
            else "background-color: #ffadad; color: #450920"
            if value == "SKIPPED"
            else ""
        ),
        subset=["status"],
    )
    .format({
        "rows": "{:,.0f}",
        "estimated_partitions": "{:,.0f}",
    })
)


empty_effective_configs = [table_name for table_name, candidates in partition_config.items() if not candidates]

if empty_effective_configs:
    raise RuntimeError(f"Aucun layout exploitable pour : {empty_effective_configs}")


print("\nTables configurées :", len(partition_config))
print("Layouts demandés :", len(layout_validation_df))
print("Layouts retenus :", int((layout_validation_df["status"] == "ACCEPTED").sum()))
print("Layouts ignorés :", int((layout_validation_df["status"] == "SKIPPED").sum()))



,table,layout,partition_columns,rows,estimated_partitions,status,reason
0,buyer,unpartitioned,None,"100,000",1,ACCEPTED,
1,carrier,unpartitioned,None,4,1,ACCEPTED,
2,cart,unpartitioned,None,0,1,ACCEPTED,
3,cart_items,unpartitioned,None,0,1,ACCEPTED,
4,category,unpartitioned,None,2,1,ACCEPTED,
5,customer,unpartitioned,None,"130,766",1,ACCEPTED,
6,customer_payment,unpartitioned,None,"100,000",1,ACCEPTED,
7,customer_shipping,unpartitioned,None,"200,001",1,ACCEPTED,
8,daily_deals,unpartitioned,None,0,1,ACCEPTED,
9,daily_deals,by_deal_date,['deal_date'],0,0,SKIPPED,empty table



Tables configurées : 27
Layouts demandés : 41
Layouts retenus : 40
Layouts ignorés : 1


### 6. génération locale des différents layouts Parquet 

In [15]:
if RUN_DIR.exists():
    raise FileExistsError(
        f"Le dossier du run existe déjà : {RUN_DIR}. "
        "Relance le bloc 1 pour obtenir un nouveau RUN_ID."
    )

RUN_DIR.mkdir(parents=True)

local_metrics = []
total_layouts = sum(len(candidates) for candidates in partition_config.values())
layout_number = 0

print("Run :", RUN_ID)
print("Dossier de sortie :", RUN_DIR)
print("Layouts à générer :", total_layouts)


for table_name, candidates in partition_config.items():
    df = prepared_tables[table_name].copy()

    for partition_columns in candidates:
        layout_number += 1
        layout_name = make_layout_name(partition_columns)
        output_dir = RUN_DIR / table_name / layout_name
        output_dir.mkdir(parents=True)

        print(f"\n[{layout_number}/{total_layouts}] {table_name} / {layout_name}")
        start = perf_counter()

        if partition_columns is None:
            df.to_parquet(
                output_dir / "data.parquet",
                engine="pyarrow",
                compression="snappy",
                index=False,
            )
            partition_count = 1
        else:
            df.to_parquet(
                output_dir,
                engine="pyarrow",
                compression="snappy",
                index=False,
                partition_cols=partition_columns,
                basename_template="part-{i}.parquet",
                max_partitions=MAX_AUDIT_PARTITIONS,
            )
            partition_count = len(df[partition_columns].drop_duplicates())

        local_write_seconds = perf_counter() - start
        parquet_files = sorted(output_dir.rglob("*.parquet"))
        object_count = len(parquet_files)
        total_bytes = sum(file.stat().st_size for file in parquet_files)
        written_rows = sum(pq.ParquetFile(file).metadata.num_rows for file in parquet_files)
        average_file_bytes = total_bytes / object_count if object_count else 0
        write_valid = object_count > 0 and written_rows == len(df)

        if not write_valid:
            raise RuntimeError(
                f"Écriture invalide pour {table_name}/{layout_name} : "
                f"{written_rows:,} lignes écrites au lieu de {len(df):,}"
            )

        local_metrics.append({
            "run_id": RUN_ID,
            "table": table_name,
            "layout": layout_name,
            "partition_columns": None if partition_columns is None else ", ".join(partition_columns),
            "rows": len(df),
            "columns": len(df.columns),
            "partition_count": partition_count,
            "object_count": object_count,
            "total_bytes": total_bytes,
            "average_file_bytes": average_file_bytes,
            "local_write_seconds": local_write_seconds,
            "written_rows": written_rows,
            "write_valid": write_valid,
            "local_path": str(output_dir),
        })

        print(
            f"{object_count:,} fichier(s) | "
            f"{partition_count:,} partition(s) | "
            f"{total_bytes:,} octets | "
            f"{local_write_seconds:.4f} s | "
            f"{written_rows:,} ligne(s) | OK"
        )


local_metrics_df = pd.DataFrame(local_metrics)

if len(local_metrics_df) != total_layouts:
    raise RuntimeError(
        f"{len(local_metrics_df)} layouts produits au lieu de {total_layouts}"
    )

if not local_metrics_df["write_valid"].all():
    raise RuntimeError("Au moins un layout Parquet est invalide.")


local_metrics_path = RESULTS_DIR / f"{RUN_ID}_local_metrics.csv"
local_metrics_df.to_csv(local_metrics_path, index=False)


print("\nGénération terminée")
print("Layouts créés :", len(local_metrics_df))
print("Fichiers Parquet :", int(local_metrics_df["object_count"].sum()))
print("Volume total :", f'{local_metrics_df["total_bytes"].sum():,}', "octets")
print("Métriques sauvegardées :", local_metrics_path)

Run : 20261012_113324
Dossier de sortie : /Users/alexandro/Desktop/Master/Mines Paris/M2/Data Management & Cloud infrastructure for AI /de-platform-template/ingestion/audit_layout/20261012_113324
Layouts à générer : 40

[1/40] buyer / unpartitioned
1 fichier(s) | 1 partition(s) | 3,252,070 octets | 0.1640 s | 100,000 ligne(s) | OK

[2/40] carrier / unpartitioned
1 fichier(s) | 1 partition(s) | 2,768 octets | 0.0021 s | 4 ligne(s) | OK

[3/40] cart / unpartitioned
1 fichier(s) | 1 partition(s) | 2,944 octets | 0.0013 s | 0 ligne(s) | OK

[4/40] cart_items / unpartitioned
1 fichier(s) | 1 partition(s) | 2,366 octets | 0.0009 s | 0 ligne(s) | OK

[5/40] category / unpartitioned
1 fichier(s) | 1 partition(s) | 3,221 octets | 0.0011 s | 2 ligne(s) | OK

[6/40] customer / unpartitioned
1 fichier(s) | 1 partition(s) | 9,895,404 octets | 0.1142 s | 130,766 ligne(s) | OK

[7/40] customer_payment / unpartitioned
1 fichier(s) | 1 partition(s) | 3,866,376 octets | 0.0268 s | 100,000 ligne(s) | OK


### 7. mesure du nombre d’objets, de leur taille et du temps d’écriture

In [16]:
layout_metrics_df = local_metrics_df[[
    "table",
    "layout",
    "rows",
    "partition_count",
    "object_count",
    "total_bytes",
    "average_file_bytes",
    "local_write_seconds",
    "write_valid",
]].sort_values(["table", "layout"]).reset_index(drop=True)

if not layout_metrics_df["write_valid"].all():
    raise RuntimeError("Au moins un layout local est invalide.")

display(
    layout_metrics_df.style.format({
        "rows": "{:,.0f}",
        "partition_count": "{:,.0f}",
        "object_count": "{:,.0f}",
        "total_bytes": "{:,.0f}",
        "average_file_bytes": "{:,.2f}",
        "local_write_seconds": "{:.6f} s",
    })
)

print("Objets totaux :", int(layout_metrics_df["object_count"].sum()))
print("Volume total :", f'{layout_metrics_df["total_bytes"].sum():,}', "octets")
print("Temps total d’écriture :", f'{layout_metrics_df["local_write_seconds"].sum():.4f}', "secondes")

,table,layout,rows,partition_count,object_count,total_bytes,average_file_bytes,local_write_seconds,write_valid
0,buyer,unpartitioned,"100,000",1,1,"3,252,070","3,252,070.00",0.163975 s,True
1,carrier,unpartitioned,4,1,1,"2,768","2,768.00",0.002066 s,True
2,cart,unpartitioned,0,1,1,"2,944","2,944.00",0.001300 s,True
3,cart_items,unpartitioned,0,1,1,"2,366","2,366.00",0.000851 s,True
4,category,unpartitioned,2,1,1,"3,221","3,221.00",0.001124 s,True
5,customer,unpartitioned,"130,766",1,1,"9,895,404","9,895,404.00",0.114221 s,True
6,customer_payment,unpartitioned,"100,000",1,1,"3,866,376","3,866,376.00",0.026755 s,True
7,customer_shipping,unpartitioned,"200,001",1,1,"7,573,765","7,573,765.00",0.053332 s,True
8,daily_deals,unpartitioned,0,1,1,"2,433","2,433.00",0.006865 s,True
9,discount,unpartitioned,0,1,1,"3,011","3,011.00",0.001137 s,True


Objets totaux : 4080
Volume total : 300,885,503 octets
Temps total d’écriture : 7.3064 secondes


### 8. upload des layouts d’audit dans le bucket personnel 

In [18]:
gcs_run_prefix = f"{AUDIT_PREFIX}/{RUN_ID}"
existing_objects = {
    obj.name: obj.size or 0
    for obj in storage_client.list_blobs(AUDIT_BUCKET_NAME, prefix=f"{gcs_run_prefix}/")
}

upload_results = []
total_layouts = len(local_metrics_df)

for layout_number, row in enumerate(local_metrics_df.itertuples(index=False), start=1):
    local_path = Path(row.local_path)
    files = sorted(local_path.rglob("*.parquet"))
    layout_prefix = f"{gcs_run_prefix}/{row.table}/{row.layout}"
    expected_objects = {}
    uploaded_count = skipped_count = 0
    start = perf_counter()

    print(f"\n[{layout_number}/{total_layouts}] {row.table} / {row.layout}")

    for file_number, local_file in enumerate(files, start=1):
        object_name = f"{layout_prefix}/{local_file.relative_to(local_path).as_posix()}"
        local_size = local_file.stat().st_size
        expected_objects[object_name] = local_size

        if object_name in existing_objects:
            if existing_objects[object_name] != local_size:
                raise RuntimeError(f"Conflit de taille : {object_name}")
            skipped_count += 1
        else:
            audit_bucket.blob(object_name).upload_from_filename(str(local_file), content_type="application/octet-stream")
            existing_objects[object_name] = local_size
            uploaded_count += 1

        if file_number % max(1, len(files) // 10) == 0 or file_number == len(files):
            print(f"  {file_number:,}/{len(files):,}")

    upload_seconds = perf_counter() - start
    remote_objects = {
        obj.name: obj.size or 0
        for obj in storage_client.list_blobs(AUDIT_BUCKET_NAME, prefix=f"{layout_prefix}/")
    }
    upload_valid = remote_objects == expected_objects

    if not upload_valid:
        raise RuntimeError(f"Upload invalide : {row.table}/{row.layout}")

    upload_results.append({
        "run_id": RUN_ID,
        "table": row.table,
        "layout": row.layout,
        "gcs_prefix": f"gs://{AUDIT_BUCKET_NAME}/{layout_prefix}/",
        "uploaded_count": uploaded_count,
        "skipped_count": skipped_count,
        "upload_seconds": upload_seconds,
        "gcs_object_count": len(remote_objects),
        "gcs_total_bytes": sum(remote_objects.values()),
        "upload_valid": upload_valid,
    })

    print(f"  {uploaded_count} envoyé(s), {skipped_count} ignoré(s) | {upload_seconds:.2f} s | OK")


upload_results_df = pd.DataFrame(upload_results)
upload_results_path = RESULTS_DIR / f"{RUN_ID}_upload_metrics.csv"
upload_results_df.to_csv(upload_results_path, index=False)

print("\nLayouts uploadés :", int(upload_results_df["upload_valid"].sum()), "/", total_layouts)
print("Objets dans GCS :", int(upload_results_df["gcs_object_count"].sum()))
print("Volume dans GCS :", f'{upload_results_df["gcs_total_bytes"].sum():,}', "octets")


[1/40] buyer / unpartitioned
  1/1
  0 envoyé(s), 1 ignoré(s) | 0.00 s | OK

[2/40] carrier / unpartitioned
  1/1
  0 envoyé(s), 1 ignoré(s) | 0.00 s | OK

[3/40] cart / unpartitioned
  1/1
  0 envoyé(s), 1 ignoré(s) | 0.00 s | OK

[4/40] cart_items / unpartitioned
  1/1
  0 envoyé(s), 1 ignoré(s) | 0.00 s | OK

[5/40] category / unpartitioned
  1/1
  0 envoyé(s), 1 ignoré(s) | 0.00 s | OK

[6/40] customer / unpartitioned
  1/1
  0 envoyé(s), 1 ignoré(s) | 0.00 s | OK

[7/40] customer_payment / unpartitioned
  1/1
  0 envoyé(s), 1 ignoré(s) | 0.00 s | OK

[8/40] customer_shipping / unpartitioned
  1/1
  0 envoyé(s), 1 ignoré(s) | 0.00 s | OK

[9/40] daily_deals / unpartitioned
  1/1
  0 envoyé(s), 1 ignoré(s) | 0.00 s | OK

[10/40] discount / unpartitioned
  1/1
  0 envoyé(s), 1 ignoré(s) | 0.00 s | OK

[11/40] order_items / unpartitioned
  1/1
  0 envoyé(s), 1 ignoré(s) | 0.00 s | OK

[12/40] orders / unpartitioned
  1/1
  0 envoyé(s), 1 ignoré(s) | 0.00 s | OK

[13/40] orders / by_o

### 9. création de tables externes BigQuery 

In [19]:
external_tables = []
generated_table_ids = set()
total_layouts = len(upload_results_df)

for layout_number, row in enumerate(upload_results_df.itertuples(index=False), start=1):
    if not row.upload_valid:
        raise RuntimeError(f"Upload non validé : {row.table}/{row.layout}")

    table_id = re.sub(r"[^A-Za-z0-9_]", "_", f"audit_{RUN_ID}__{row.table}__{row.layout}")
    full_table_id = f"{PROJECT_ID}.{BIGQUERY_DATASET}.{table_id}"

    if table_id in generated_table_ids:
        raise RuntimeError(f"Collision de nom BigQuery : {table_id}")

    generated_table_ids.add(table_id)
    gcs_prefix = row.gcs_prefix.rstrip("/") + "/"
    source_uri = f"{gcs_prefix}*"

    external_config = bigquery.ExternalConfig("PARQUET")
    external_config.source_uris = [source_uri]
    external_config.autodetect = True

    if row.layout != "unpartitioned":
        hive_options = bigquery.HivePartitioningOptions()
        hive_options.mode = "STRINGS"
        hive_options.require_partition_filter = False
        hive_options.source_uri_prefix = gcs_prefix
        external_config.hive_partitioning = hive_options

    table = bigquery.Table(full_table_id)
    table.external_data_configuration = external_config

    try:
        existing_table = bigquery_client.get_table(full_table_id)
        existing_uris = sorted(existing_table.external_data_configuration.source_uris)

        if existing_uris != [source_uri]:
            raise RuntimeError(f"La table existe avec une autre source : {full_table_id}")

        status = "EXISTS"
    except NotFound:
        bigquery_client.create_table(table)
        status = "CREATED"

    external_tables.append({
        "run_id": RUN_ID,
        "table": row.table,
        "layout": row.layout,
        "bigquery_table_id": full_table_id,
        "gcs_prefix": gcs_prefix,
        "creation_status": status,
    })

    print(f"[{layout_number}/{total_layouts}] {row.table} / {row.layout} : {status}")


external_tables_df = pd.DataFrame(external_tables)
external_tables_path = RESULTS_DIR / f"{RUN_ID}_external_tables.csv"
external_tables_df.to_csv(external_tables_path, index=False)

if len(external_tables_df) != total_layouts:
    raise RuntimeError("Toutes les tables externes n'ont pas été référencées.")

print("\nTables externes référencées :", len(external_tables_df))
print(external_tables_df["creation_status"].value_counts())

[1/40] buyer / unpartitioned : CREATED
[2/40] carrier / unpartitioned : CREATED
[3/40] cart / unpartitioned : CREATED
[4/40] cart_items / unpartitioned : CREATED
[5/40] category / unpartitioned : CREATED
[6/40] customer / unpartitioned : CREATED
[7/40] customer_payment / unpartitioned : CREATED
[8/40] customer_shipping / unpartitioned : CREATED
[9/40] daily_deals / unpartitioned : CREATED
[10/40] discount / unpartitioned : CREATED
[11/40] order_items / unpartitioned : CREATED
[12/40] orders / unpartitioned : CREATED
[13/40] orders / by_order_date : CREATED
[14/40] payment / unpartitioned : CREATED
[15/40] payment / by_method : CREATED
[16/40] payment_details / unpartitioned : CREATED
[17/40] product / unpartitioned : CREATED
[18/40] product / by_category_id : CREATED
[19/40] product_images / unpartitioned : CREATED
[20/40] product_reviews / unpartitioned : CREATED
[21/40] returns / unpartitioned : CREATED
[22/40] review / unpartitioned : CREATED
[23/40] review / by_rating : CREATED
[24

### 10. exécution de requêtes `SELECT` représentatives 

In [34]:
# ============================================================
# STEP 10 — REQUÊTES SELECT BIGQUERY
# ============================================================

from urllib.parse import quote


def sql_literal(value, field_type):
    if field_type in {"INTEGER", "INT64", "FLOAT", "FLOAT64", "NUMERIC", "BIGNUMERIC"}:
        return str(value)

    if field_type in {"BOOLEAN", "BOOL"}:
        return "TRUE" if bool(value) else "FALSE"

    escaped = str(value).replace("'", "''")

    if field_type == "DATE":
        return f"DATE '{escaped}'"
    if field_type == "DATETIME":
        return f"DATETIME '{escaped}'"
    if field_type == "TIMESTAMP":
        return f"TIMESTAMP '{escaped}'"

    return f"'{escaped}'"


# Colonnes physiquement utilisées par chaque layout
layout_partition_columns = {
    (table_name, make_layout_name(columns)): columns or []
    for table_name, candidates in partition_config.items()
    for columns in candidates
}


# Même filtre logique pour tous les layouts d'une table
query_filters = {}

for table_name, candidates in partition_config.items():
    partitioned_candidates = [
        candidate for candidate in candidates
        if candidate is not None
    ]

    filter_columns = (
        max(partitioned_candidates, key=len)
        if partitioned_candidates
        else []
    )

    df = prepared_tables[table_name]

    if not filter_columns or df.empty:
        query_filters[table_name] = {}
        continue

    combinations = df[filter_columns].dropna().value_counts()

    if combinations.empty:
        query_filters[table_name] = {}
        continue

    selected_values = combinations.index[0]

    if not isinstance(selected_values, tuple):
        selected_values = (selected_values,)

    query_filters[table_name] = dict(
        zip(filter_columns, selected_values)
    )


# Colonne BigQuery commune à tous les layouts,
# non partitionnée et non sensible
probe_columns = {}

for table_name in partition_config:
    table_references = external_tables_df[
        external_tables_df["table"] == table_name
    ]

    common_columns = None
    ordered_columns = []

    for table_reference in table_references.itertuples(index=False):
        bq_table = bigquery_client.get_table(
            table_reference.bigquery_table_id
        )

        schema_columns = [
            field.name
            for field in bq_table.schema
        ]

        if not ordered_columns:
            ordered_columns = schema_columns

        common_columns = (
            set(schema_columns)
            if common_columns is None
            else common_columns & set(schema_columns)
        )

    all_partition_columns = {
        column
        for candidate in partition_config[table_name]
        if candidate is not None
        for column in candidate
    }

    candidates = [
        column
        for column in ordered_columns
        if column in common_columns
        and column not in all_partition_columns
        and not column.lower().startswith("unnamed")
        and not is_sensitive_column(column)
    ]

    if not candidates:
        candidates = [
            column
            for column in ordered_columns
            if column in common_columns
            and column not in all_partition_columns
            and not column.lower().startswith("unnamed")
        ]

    if not candidates:
        raise RuntimeError(
            f"Aucune colonne de test commune pour {table_name}"
        )

    probe_columns[table_name] = candidates[0]

# Vérification du run utilisé
audit_run_ids = local_metrics_df["run_id"].unique()

if len(audit_run_ids) != 1:
    raise RuntimeError(
        f"Plusieurs runs locaux détectés : {audit_run_ids}"
    )

AUDIT_RUN_ID = audit_run_ids[0]

if set(upload_results_df["run_id"]) != {AUDIT_RUN_ID}:
    raise RuntimeError(
        "Le run GCS ne correspond pas au run local."
    )

if set(external_tables_df["run_id"]) != {AUDIT_RUN_ID}:
    raise RuntimeError(
        "Le run BigQuery ne correspond pas au run local."
    )

print("Run d’audit utilisé :", AUDIT_RUN_ID)


# Exécution des requêtes
bq_metrics = []
total_queries = len(external_tables_df)

for query_number, row in enumerate(
    external_tables_df.itertuples(index=False),
    start=1,
):
    bq_table = bigquery_client.get_table(
        row.bigquery_table_id
    )

    schema_types = {
        field.name: field.field_type
        for field in bq_table.schema
    }

    filters = query_filters[row.table]
    probe_column = probe_columns[row.table]

    if probe_column not in schema_types:
        raise RuntimeError(
            f"Colonne {probe_column} absente de "
            f"{row.bigquery_table_id}"
        )

    physical_partition_columns = (
        layout_partition_columns[
            (row.table, row.layout)
        ]
    )

    conditions = []

    for column, logical_value in filters.items():
        query_value = (
            quote(str(logical_value), safe="")
            if column in physical_partition_columns
            else logical_value
        )

        conditions.append(
            f"`{column}` = "
            f"{sql_literal(query_value, schema_types[column])}"
        )

    where_clause = (
        f"WHERE {' AND '.join(conditions)}"
        if conditions
        else ""
    )

    query = f"""
        SELECT
            COUNT(*) AS selected_rows,
            SUM(
                BYTE_LENGTH(
                    COALESCE(
                        CAST(`{probe_column}` AS STRING),
                        ''
                    )
                )
            ) AS probe_value
        FROM `{row.bigquery_table_id}`
        {where_clause}
    """

    print(
        f"[{query_number}/{total_queries}] "
        f"{row.table} / {row.layout} | "
        f"probe={probe_column}"
    )

    job_config = bigquery.QueryJobConfig(
        use_query_cache=False
    )

    try:
        query_job = bigquery_client.query(
            query,
            job_config=job_config,
            location=GCP_REGION,
        )

        result = list(query_job.result())

    except Exception:
        print("Requête en erreur :")
        print(query)
        raise

    elapsed_ms = (
        query_job.ended - query_job.started
    ).total_seconds() * 1000

    selected_rows = result[0]["selected_rows"]
    probe_value = result[0]["probe_value"]

    bq_metrics.append({
        "run_id": AUDIT_RUN_ID,
        "table": row.table,
        "layout": row.layout,
        "bigquery_table_id": row.bigquery_table_id,
        "query_filter": json.dumps(
            filters,
            default=str,
        ),
        "probe_column": probe_column,
        "probe_value": probe_value,
        "selected_rows": selected_rows,
        "query_elapsed_ms": elapsed_ms,
        "query_bytes_processed": (
            query_job.total_bytes_processed or 0
        ),
        "bigquery_job_id": query_job.job_id,
        "query": " ".join(query.split()),
    })

    print(
        f"  {selected_rows:,} ligne(s) | "
        f"{elapsed_ms:.2f} ms | "
        f"{query_job.total_bytes_processed or 0:,} octets"
    )


bq_metrics_df = pd.DataFrame(bq_metrics)

bq_metrics_path = (
    RESULTS_DIR
    / f"{AUDIT_RUN_ID}_bigquery_metrics.csv"
)

bq_metrics_df.to_csv(
    bq_metrics_path,
    index=False,
)

if len(bq_metrics_df) != total_queries:
    raise RuntimeError(
        "Toutes les requêtes BigQuery "
        "n'ont pas été exécutées."
    )

print(
    "\nRequêtes exécutées :",
    len(bq_metrics_df),
)

print(
    "Métriques sauvegardées :",
    bq_metrics_path,
)

Run d’audit utilisé : 20261012_113324
[1/40] buyer / unpartitioned | probe=buyer_id
  100,000 ligne(s) | 264.00 ms | 3,000,024 octets
[2/40] carrier / unpartitioned | probe=carrier_id
  4 ligne(s) | 191.00 ms | 32 octets
[3/40] cart / unpartitioned | probe=cart_id
  0 ligne(s) | 194.00 ms | 0 octets
[4/40] cart_items / unpartitioned | probe=cart_id
  0 ligne(s) | 179.00 ms | 0 octets
[5/40] category / unpartitioned | probe=category_id
  2 ligne(s) | 203.00 ms | 16 octets
[6/40] customer / unpartitioned | probe=c_id
  130,766 ligne(s) | 405.00 ms | 3,984,531 octets
[7/40] customer_payment / unpartitioned | probe=payment_id
  100,000 ligne(s) | 251.00 ms | 800,000 octets
[8/40] customer_shipping / unpartitioned | probe=c_id
  200,001 ligne(s) | 286.00 ms | 6,000,064 octets
[9/40] daily_deals / unpartitioned | probe=p_id
  0 ligne(s) | 177.00 ms | 0 octets
[10/40] discount / unpartitioned | probe=discount_id
  0 ligne(s) | 196.00 ms | 0 octets
[11/40] order_items / unpartitioned | probe=o

### 11. comparaison des coûts et performances ;

In [35]:
final_metrics_df = (
    local_metrics_df
    .merge(upload_results_df, on=["run_id", "table", "layout"], how="left", validate="one_to_one")
    .merge(bq_metrics_df, on=["run_id", "table", "layout"], how="left", validate="one_to_one")
)

if len(final_metrics_df) != len(local_metrics_df):
    raise RuntimeError("Le nombre de layouts a changé pendant les jointures.")

required_metrics = ["gcs_object_count", "gcs_total_bytes", "query_elapsed_ms", "query_bytes_processed"]

if final_metrics_df[required_metrics].isna().any().any():
    raise RuntimeError("Certaines métriques sont manquantes.")

# Toutes les versions d'une table doivent renvoyer le même résultat
invalid_results = (
    final_metrics_df.groupby("table")["selected_rows"].nunique()
    .loc[lambda values: values > 1]
)

if not invalid_results.empty:
    raise RuntimeError(f"Résultats SQL incohérents : {invalid_results.index.tolist()}")


final_metrics_df["average_file_bytes"] = (
    final_metrics_df["gcs_total_bytes"] / final_metrics_df["gcs_object_count"]
)

final_metrics_df["storage_cost_usd_month"] = (
    final_metrics_df["gcs_total_bytes"] / BYTES_PER_GIB
    * STANDARD_STORAGE_USD_PER_GIB_MONTH
)

final_metrics_df["class_a_write_cost_usd"] = (
    final_metrics_df["gcs_object_count"] / 1000
    * CLASS_A_USD_PER_1000_OPERATIONS
)


comparison_columns = [
    "table",
    "layout",
    "partition_count",
    "gcs_object_count",
    "gcs_total_bytes",
    "average_file_bytes",
    "class_a_write_cost_usd",
    "storage_cost_usd_month",
    "query_elapsed_ms",
    "query_bytes_processed",
]

lower_is_better = [
    "gcs_object_count",
    "gcs_total_bytes",
    "class_a_write_cost_usd",
    "storage_cost_usd_month",
    "query_elapsed_ms",
    "query_bytes_processed",
]

green_style = {"background-color": "#b7e4c7", "color": "#081c15", "font-weight": "bold"}
red_style = {"background-color": "#ffadad", "color": "#450920", "font-weight": "bold"}

comparison_df = final_metrics_df[comparison_columns].sort_values(["table", "layout"]).reset_index(drop=True)

for table_name in comparison_df["table"].unique():
    table_df = comparison_df[comparison_df["table"] == table_name].reset_index(drop=True)
    styled = table_df.style

    for metric in lower_is_better:
        minimum, maximum = table_df[metric].min(), table_df[metric].max()
        best_indices = table_df.index[table_df[metric] == minimum].tolist()
        styled = styled.set_properties(subset=pd.IndexSlice[best_indices, [metric]], **green_style)

        if minimum != maximum:
            worst_indices = table_df.index[table_df[metric] == maximum].tolist()
            styled = styled.set_properties(subset=pd.IndexSlice[worst_indices, [metric]], **red_style)

    print(f"\n{table_name}")

    display(
        styled.format({
            "partition_count": "{:,.0f}",
            "gcs_object_count": "{:,.0f}",
            "gcs_total_bytes": "{:,.0f}",
            "average_file_bytes": "{:,.2f}",
            "class_a_write_cost_usd": "${:.12e}",
            "storage_cost_usd_month": "${:.12e}",
            "query_elapsed_ms": "{:.2f} ms",
            "query_bytes_processed": "{:,.0f}",
        })
    )


final_metrics_path = RESULTS_DIR / f"{RUN_ID}_final_metrics.csv"
final_metrics_df.to_csv(final_metrics_path, index=False)

print("\nLayouts comparés :", len(final_metrics_df))
print("Résultats SQL cohérents : OK")
print("Métriques finales sauvegardées :", final_metrics_path)


buyer


,table,layout,partition_count,gcs_object_count,gcs_total_bytes,average_file_bytes,class_a_write_cost_usd,storage_cost_usd_month,query_elapsed_ms,query_bytes_processed
0,buyer,unpartitioned,1,1,"3,252,070","3,252,070.00",$5.000000000000e-06,$6.057452410460e-05,264.00 ms,"3,000,024"



carrier


,table,layout,partition_count,gcs_object_count,gcs_total_bytes,average_file_bytes,class_a_write_cost_usd,storage_cost_usd_month,query_elapsed_ms,query_bytes_processed
0,carrier,unpartitioned,1,1,"2,768","2,768.00",$5.000000000000e-06,$5.155801773071e-08,191.00 ms,32



cart


...


subscription


,table,layout,partition_count,gcs_object_count,gcs_total_bytes,average_file_bytes,class_a_write_cost_usd,storage_cost_usd_month,query_elapsed_ms,query_bytes_processed
0,subscription,by_start_date,366,366,"3,738,036","10,213.21",$1.830000000000e-03,$6.962634623051e-05,293.00 ms,"1,360"
1,subscription,unpartitioned,1,1,"2,083,977","2,083,977.00",$5.000000000000e-06,$3.881709650159e-05,242.00 ms,"1,001,820"



wishlist_item


,table,layout,partition_count,gcs_object_count,gcs_total_bytes,average_file_bytes,class_a_write_cost_usd,storage_cost_usd_month,query_elapsed_ms,query_bytes_processed
0,wishlist_item,unpartitioned,1,1,"944,699","944,699.00",$5.000000000000e-06,$1.759639009833e-05,188.00 ms,"297,720"



Layouts comparés : 40
Résultats SQL cohérents : OK
Métriques finales sauvegardées : /Users/alexandro/Desktop/Master/Mines Paris/M2/Data Management & Cloud infrastructure for AI /de-platform-template/ingestion/audit_results/20261012_121604_final_metrics.csv


### 12. choix d’un seul layout final par table 

In [32]:
available_layouts = (
    final_metrics_df.groupby("table")["layout"]
    .apply(list)
    .to_dict()
)

if not FINAL_LAYOUT_CONFIG_PATH.exists():
    template = {
        table_name: {
            "selected_layout": "unpartitioned",
            "available_layouts": layouts,
            "reason": "TO_COMPLETE",
        }
        for table_name, layouts in available_layouts.items()
    }

    FINAL_LAYOUT_CONFIG_PATH.write_text(
        json.dumps(template, indent=4, ensure_ascii=False),
        encoding="utf-8",
    )

    raise RuntimeError(
        f"Fichier créé : {FINAL_LAYOUT_CONFIG_PATH}. "
        "Complète les décisions puis relance cette cellule."
    )


with open(FINAL_LAYOUT_CONFIG_PATH, "r", encoding="utf-8") as file:
    final_layout_config = json.load(file)


missing_tables = set(available_layouts) - set(final_layout_config)
unknown_tables = set(final_layout_config) - set(available_layouts)

if missing_tables:
    raise ValueError(f"Décisions manquantes : {sorted(missing_tables)}")

if unknown_tables:
    raise ValueError(f"Tables inconnues : {sorted(unknown_tables)}")


selected_rows = []

for table_name, decision in final_layout_config.items():
    selected_layout = decision.get("selected_layout")
    reason = decision.get("reason", "").strip()

    if selected_layout not in available_layouts[table_name]:
        raise ValueError(
            f"Layout invalide pour {table_name} : {selected_layout}. "
            f"Disponibles : {available_layouts[table_name]}"
        )

    if not reason or reason == "TO_COMPLETE":
        raise ValueError(f"Justification manquante pour {table_name}")

    match = final_metrics_df[
        (final_metrics_df["table"] == table_name)
        & (final_metrics_df["layout"] == selected_layout)
    ]

    if len(match) != 1:
        raise RuntimeError(f"Décision ambiguë pour {table_name}")

    selected_row = match.iloc[0].to_dict()
    selected_row["decision_reason"] = reason
    selected_rows.append(selected_row)


selected_layouts_df = pd.DataFrame(selected_rows).sort_values("table").reset_index(drop=True)

display(
    selected_layouts_df[[
        "table",
        "layout",
        "gcs_object_count",
        "gcs_total_bytes",
        "class_a_write_cost_usd",
        "storage_cost_usd_month",
        "query_elapsed_ms",
        "query_bytes_processed",
        "decision_reason",
    ]].style.format({
        "gcs_object_count": "{:,.0f}",
        "gcs_total_bytes": "{:,.0f}",
        "class_a_write_cost_usd": "${:.12e}",
        "storage_cost_usd_month": "${:.12e}",
        "query_elapsed_ms": "{:.2f} ms",
        "query_bytes_processed": "{:,.0f}",
    })
)

selected_layouts_path = RESULTS_DIR / f"{RUN_ID}_selected_layouts.csv"
selected_layouts_df.to_csv(selected_layouts_path, index=False)

print("\nDécisions validées :", len(selected_layouts_df))
print("Décisions sauvegardées :", selected_layouts_path)

RuntimeError: Fichier créé : /Users/alexandro/Desktop/Master/Mines Paris/M2/Data Management & Cloud infrastructure for AI /de-platform-template/ingestion/final_bronze_layout.json. Complète les décisions puis relance cette cellule.

### 13. déploiement des fichiers retenus dans le bucket Bronze du groupe 

In [ ]:
if GROUP_BUCKET_NAME == "REPLACE_WITH_GROUP_BUCKET_NAME":
    raise RuntimeError("Configure GROUP_BUCKET_NAME dans le bloc 1.")

group_bucket = storage_client.get_bucket(GROUP_BUCKET_NAME)
deployment_results = []
total_tables = len(selected_layouts_df)

for table_number, row in enumerate(selected_layouts_df.itertuples(index=False), start=1):
    local_path = Path(row.local_path)
    local_files = sorted(local_path.rglob("*.parquet"))
    target_prefix = f"{FINAL_BRONZE_PREFIX}/{row.table}"

    if not local_files:
        raise RuntimeError(f"Aucun fichier local pour {row.table}/{row.layout}")

    expected_objects = {
        f"{target_prefix}/{file.relative_to(local_path).as_posix()}": file.stat().st_size
        for file in local_files
    }

    existing_objects = {
        obj.name: obj.size or 0
        for obj in storage_client.list_blobs(GROUP_BUCKET_NAME, prefix=f"{target_prefix}/")
    }

    unexpected_objects = set(existing_objects) - set(expected_objects)

    if unexpected_objects:
        raise RuntimeError(
            f"{row.table} contient déjà des objets inattendus : "
            f"{sorted(unexpected_objects)[:5]}"
        )

    uploaded_count = skipped_count = 0
    start = perf_counter()

    print(f"\n[{table_number}/{total_tables}] {row.table} / {row.layout}")

    for file_number, local_file in enumerate(local_files, start=1):
        object_name = f"{target_prefix}/{local_file.relative_to(local_path).as_posix()}"
        local_size = local_file.stat().st_size

        if object_name in existing_objects:
            if existing_objects[object_name] != local_size:
                raise RuntimeError(f"Conflit de taille : {object_name}")
            skipped_count += 1
        else:
            group_bucket.blob(object_name).upload_from_filename(
                str(local_file),
                content_type="application/octet-stream",
            )
            uploaded_count += 1

        if file_number % max(1, len(local_files) // 10) == 0 or file_number == len(local_files):
            print(f"  {file_number:,}/{len(local_files):,}")

    deployment_results.append({
        "table": row.table,
        "layout": row.layout,
        "local_path": str(local_path),
        "gcs_prefix": f"gs://{GROUP_BUCKET_NAME}/{target_prefix}/",
        "expected_object_count": len(expected_objects),
        "expected_total_bytes": sum(expected_objects.values()),
        "uploaded_count": uploaded_count,
        "skipped_count": skipped_count,
        "deployment_seconds": perf_counter() - start,
    })

    print(f"  {uploaded_count} envoyé(s), {skipped_count} déjà présent(s)")


deployment_results_df = pd.DataFrame(deployment_results)
deployment_results_path = RESULTS_DIR / f"{RUN_ID}_bronze_deployment.csv"
deployment_results_df.to_csv(deployment_results_path, index=False)

print("\nTables déployées :", len(deployment_results_df))
print("Rapport sauvegardé :", deployment_results_path)

### 14. vérification du contenu local par rapport au contenu GCS

In [ ]:
def local_md5(file_path):
    digest = hashlib.md5()

    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return base64.b64encode(digest.digest()).decode()


verification_results = []

for row in deployment_results_df.itertuples(index=False):
    local_path = Path(row.local_path)
    target_prefix = row.gcs_prefix.replace(f"gs://{GROUP_BUCKET_NAME}/", "").rstrip("/")

    local_objects = {
        f"{target_prefix}/{file.relative_to(local_path).as_posix()}": {
            "size": file.stat().st_size,
            "md5": local_md5(file),
        }
        for file in local_path.rglob("*.parquet")
    }

    remote_objects = {
        obj.name: {
            "size": obj.size or 0,
            "md5": obj.md5_hash,
        }
        for obj in storage_client.list_blobs(GROUP_BUCKET_NAME, prefix=f"{target_prefix}/")
    }

    missing = set(local_objects) - set(remote_objects)
    unexpected = set(remote_objects) - set(local_objects)

    size_mismatches = [
        name for name in set(local_objects) & set(remote_objects)
        if local_objects[name]["size"] != remote_objects[name]["size"]
    ]

    checksum_mismatches = [
        name for name in set(local_objects) & set(remote_objects)
        if local_objects[name]["md5"] != remote_objects[name]["md5"]
    ]

    valid = not missing and not unexpected and not size_mismatches and not checksum_mismatches
    object_count = len(remote_objects)
    total_bytes = sum(item["size"] for item in remote_objects.values())

    verification_results.append({
        "table": row.table,
        "layout": row.layout,
        "object_count": object_count,
        "total_bytes": total_bytes,
        "missing_objects": len(missing),
        "unexpected_objects": len(unexpected),
        "size_mismatches": len(size_mismatches),
        "checksum_mismatches": len(checksum_mismatches),
        "deployment_valid": valid,
    })

    print(f"{row.table} / {row.layout} : {'OK' if valid else 'ERROR'}")


final_bronze_df = pd.DataFrame(verification_results)

final_bronze_df["class_a_write_cost_usd"] = (
    final_bronze_df["object_count"] / 1000
    * CLASS_A_USD_PER_1000_OPERATIONS
)

final_bronze_df["storage_cost_usd_month"] = (
    final_bronze_df["total_bytes"] / BYTES_PER_GIB
    * STANDARD_STORAGE_USD_PER_GIB_MONTH
)

display(
    final_bronze_df.style
    .applymap(
        lambda value: (
            "background-color: #b7e4c7; color: #081c15"
            if value is True
            else "background-color: #ffadad; color: #450920"
            if value is False
            else ""
        ),
        subset=["deployment_valid"],
    )
    .format({
        "object_count": "{:,.0f}",
        "total_bytes": "{:,.0f}",
        "class_a_write_cost_usd": "${:.12e}",
        "storage_cost_usd_month": "${:.12e}",
    })
)

if not final_bronze_df["deployment_valid"].all():
    invalid_tables = final_bronze_df.loc[
        ~final_bronze_df["deployment_valid"],
        "table",
    ].tolist()

    raise RuntimeError(f"Déploiements invalides : {invalid_tables}")


verification_path = RESULTS_DIR / f"{RUN_ID}_bronze_verification.csv"
final_bronze_df.to_csv(verification_path, index=False)

print("\nVérification finale : OK")
print("Tables validées :", len(final_bronze_df))
print("Objets totaux :", int(final_bronze_df["object_count"].sum()))
print("Volume total :", f'{final_bronze_df["total_bytes"].sum():,}', "octets")
print("Coût de création :", f'${final_bronze_df["class_a_write_cost_usd"].sum():.12f}')
print("Coût mensuel de stockage :", f'${final_bronze_df["storage_cost_usd_month"].sum():.12f}')
print("Rapport sauvegardé :", verification_path)